# **Semana 8: Captura de Datos - Web Scraping y APIs (NB1 - Conceptual)**

## **Propósito de la Sesión**
Extraer datos de fuentes externas (webs y APIs) para alimentar bases de datos y sistemas de IA, utilizando técnicas fundamentales de captura en Python.

### **Objetivos de Aprendizaje**
Al finalizar este notebook, el estudiante será capaz de:
1.  **Comprender** la importancia de la captura de datos en el ciclo de vida de un proyecto de IA.
2.  **Aplicar** técnicas de web scraping con `requests` y `BeautifulSoup` para extraer información de páginas web estáticas.
3.  **Consumir** APIs REST públicas de manera programática para obtener datos estructurados.
4.  **Almacenar** los datos capturados en un DataFrame de Pandas y persistirlos en una base de datos SQLite local.
5.  **Reflexionar** sobre las consideraciones éticas y legales de la captura de datos.

## **Configuración Inicial**

Importamos las librerías necesarias y configuramos el entorno. En este notebook, usaremos:
*   `requests`: Para hacer peticiones HTTP a páginas web y APIs.
*   `BeautifulSoup (bs4)`: Para parsear y extraer información de HTML.
*   `pandas`: Para estructurar y manipular los datos capturados.
*   `sqlite3`: Para guardar los datos en una base de datos local.
*   `matplotlib` y `seaborn`: Para visualizaciones simples.

In [ ]:
# Instalación de librerías necesarias (si no están instaladas en el entorno de Colab)
# En Colab, requests, pandas y matplotlib ya vienen instaladas.
!pip install beautifulsoup4 --quiet

# Importación de librerías
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
from datetime import datetime

# Configuración de visualización
sns.set_style("whitegrid")
pd.set_option('display.max_colwidth', 200)

print("Librerías importadas correctamente.")

---
## **Actividad 1: Web Scraping de una Página Simple**

En esta actividad simularemos la extracción de datos de un sitio web. Usaremos una página de pruebas segura y legal: **Quotes to Scrape** (`http://quotes.toscrape.com`).

### **Concepto Clave:**
El web scraping consiste en descargar el contenido HTML de una página y luego extraer información específica de él, localizando los elementos mediante sus etiquetas, clases o IDs.

### **Caso de uso en IA:**
Esta técnica es fundamental para crear corpus de texto para entrenar Modelos de Lenguaje (LLMs). Por ejemplo, el dataset **Common Crawl**, usado para entrenar GPT, se construyó mediante scraping masivo de la web.

In [ ]:
# 1. Hacer la petición HTTP a la página
url = 'http://quotes.toscrape.com'
print(f"Obteniendo datos de: {url}")

response = requests.get(url)

# Verificamos que la petición fue exitosa (código 200)
if response.status_code == 200:
    print("Página descargada correctamente.")
else:
    print(f"Error al descargar la página. Código: {response.status_code}")

# 2. Parsear el contenido HTML con BeautifulSoup
soup = BeautifulSoup(response.text, 'html.parser')
print("HTML parseado.")

# 3. Extraer la información: buscaremos todas las citas (cada una está dentro de un div con clase 'quote')
quotes_data = []
quotes_elements = soup.select('.quote') # Selector CSS: elementos con clase 'quote'

print(f"\nEncontradas {len(quotes_elements)} citas en la primera página.\n")

for quote in quotes_elements:
    # Extraemos el texto de la cita
    text = quote.select_one('.text').get_text()
    # Extraemos el autor
    author = quote.select_one('.author').get_text()
    # Extraemos las etiquetas
    tags = [tag.get_text() for tag in quote.select('.tag')]

    quotes_data.append({
        'quote': text,
        'author': author,
        'tags': ', '.join(tags)  # Convertimos la lista de tags en un string
    })

# 4. Convertir los datos a un DataFrame de Pandas
df_quotes = pd.DataFrame(quotes_data)
print("Datos extraídos y cargados en DataFrame:")
display(df_quotes)

#### **Análisis rápido de los datos extraídos**
Podemos usar el DataFrame para hacer preguntas sencillas sobre los datos.

In [ ]:
# ¿Cuántas citas hay por autor?
author_counts = df_quotes['author'].value_counts()
print("Número de citas por autor:")
print(author_counts)

# Visualizamos los autores con más citas
plt.figure(figsize=(8, 4))
author_counts.head(5).plot(kind='bar', color='skyblue')
plt.title('Top 5 Autores con más Citas (Primera Página)')
plt.xlabel('Autor')
plt.ylabel('Número de Citas')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## **Actividad 2: Consumir una API Pública**

Ahora, en lugar de extraer HTML, nos conectaremos directamente a una API que nos devuelve datos en formato JSON (el formato estándar para intercambio de datos).

Usaremos la API pública de pruebas **JSONPlaceholder** (`https://jsonplaceholder.typicode.com`).

### **Concepto Clave:**
Una API (Interfaz de Programación de Aplicaciones) actúa como un intermediario que permite a dos aplicaciones comunicarse. Para los datos, es una fuente mucho más limpia y estructurada que el scraping, ya que los datos son devueltos en un formato (generalmente JSON) listo para ser consumido.

### **Conexión con IA:**
Muchos servicios de IA (como los de reconocimiento de imágenes, análisis de sentimiento o generación de texto) se ofrecen a través de APIs. Consumir estas APIs es la forma principal de integrar la IA en aplicaciones.

In [ ]:
# 1. Definir la URL del endpoint de la API
api_url = 'https://jsonplaceholder.typicode.com/posts'

# 2. Hacer la petición GET
print(f"Consumiendo API: {api_url}")
response = requests.get(api_url)

# 3. Verificar el estado de la respuesta
if response.status_code == 200:
    print("Datos obtenidos correctamente.")
else:
    print(f"Error en la API. Código: {response.status_code}")

# 4. Convertir la respuesta (JSON) a una estructura de Python (lista de diccionarios)
posts_data = response.json()
print(f"Se obtuvieron {len(posts_data)} publicaciones.")

# 5. Mostrar un ejemplo del primer post
print("\nEjemplo del primer post:")
print(json.dumps(posts_data[0], indent=2))

# 6. Crear un DataFrame con los primeros 10 posts para visualizarlos
df_posts = pd.DataFrame(posts_data[:10])
print("\nPrimeros 10 posts en DataFrame:")
display(df_posts)

#### **Análisis simple de los datos de la API**
Podemos explorar los datos, por ejemplo, viendo cuántos posts ha escrito cada usuario (userId).

In [ ]:
# Contar posts por usuario
df_all_posts = pd.DataFrame(posts_data)
user_post_counts = df_all_posts['userId'].value_counts().sort_index()

plt.figure(figsize=(10, 4))
user_post_counts.plot(kind='bar', color='salmon')
plt.title('Número de Posts por Usuario (Usuario ID)')
plt.xlabel('ID de Usuario')
plt.ylabel('Cantidad de Posts')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## **Actividad 3: Almacenamiento en SQLite**

Una vez capturados los datos, el siguiente paso es almacenarlos. Aquí guardaremos los datos scrapeados y los de la API en una base de datos SQLite local. Esto simula el proceso de guardar datos en un *Data Lake* o *Data Warehouse*.

### **Concepto Clave:**
Persistir los datos nos permite no tener que capturarlos de nuevo cada vez que los necesitemos. SQLite es una base de datos ligera, perfecta para desarrollo local y experimentación.

In [ ]:
# 1. Conectar a la base de datos (se creará si no existe)
conn = sqlite3.connect('captura_datos.db')
cursor = conn.cursor()
print("Base de datos 'captura_datos.db' creada/conectada.")

# 2. Guardar el DataFrame de las citas (df_quotes) en una tabla llamada 'quotes'
df_quotes.to_sql('quotes', conn, if_exists='replace', index=False)
print("Tabla 'quotes' creada y datos guardados.")

# 3. Guardar el DataFrame de los posts (df_all_posts) en una tabla llamada 'posts'
df_all_posts.to_sql('posts', conn, if_exists='replace', index=False)
print("Tabla 'posts' creada y datos guardados.")

# 4. Verificar que se guardaron correctamente
print("\n--- Verificación de datos ---")

# Consultar las primeras filas de la tabla 'quotes'
query_result = pd.read_sql_query("SELECT * FROM quotes LIMIT 3", conn)
print("\nPrimeras 3 filas de la tabla 'quotes':")
display(query_result)

# Consultar cuántos posts hay en la tabla 'posts'
post_count = pd.read_sql_query("SELECT COUNT(*) as total FROM posts", conn)
print(f"\nNúmero total de posts en la tabla 'posts': {post_count['total'][0]}")

# 5. Cerrar la conexión
conn.close()
print("\nConexión a la base de datos cerrada.")

---
## **Ejercicios Prácticos para el Estudiante**

Ahora es tu turno. Intenta resolver los siguientes ejercicios basándote en lo aprendido.

#### **Ejercicio 1: Scraping de Múltiples Páginas**
Modifica el código de la **Actividad 1** para extraer todas las citas de las primeras 5 páginas de `http://quotes.toscrape.com`. Pista: Observa cómo cambia la URL al hacer clic en el botón "Next". Almacena el resultado en un DataFrame y muéstralo.

In [ ]:
# Escribe aquí tu solución para el Ejercicio 1
import requests
from bs4 import BeautifulSoup
import pandas as pd

# --- INICIO DE TU CÓDIGO ---
url_base = 'http://quotes.toscrape.com'
# ...

# --- FIN DE TU CÓDIGO ---


#### **Ejercicio 2: Explorar otro Endpoint de la API**
La API de JSONPlaceplaceholer tiene más endpoints, como `/users` o `/comments`. Consume el endpoint `/users` (`https://jsonplaceholder.typicode.com/users`), carga los datos en un DataFrame y muestra los nombres de usuario (`name`) y sus emails en una tabla.

In [ ]:
# Escribe aquí tu solución para el Ejercicio 2

# --- INICIO DE TU CÓDIGO ---
api_url_users = 'https://jsonplaceholder.typicode.com/users'
# ...

# --- FIN DE TU CÓDIGO ---

#### **Ejercicio 3: Simular una Petición con Parámetros**
Las APIs suelen aceptar parámetros para filtrar datos. Por ejemplo, para obtener solo los posts del usuario con ID 1, puedes usar: `https://jsonplaceholder.typicode.com/posts?userId=1`. 

Haz una petición a la API para obtener los posts del `userId=5`. Convierte la respuesta en un DataFrame y cuenta cuántos posts tiene este usuario.

In [ ]:
# Escribe aquí tu solución para el Ejercicio 3

# --- INICIO DE TU CÓDIGO ---
# ...

# --- FIN DE TU CÓDIGO ---

#### **Ejercicio 4: Combinar Datos de Scraping y API**
¿Podríamos enriquecer los datos de las citas con información de sus autores? Imagina que tienes un archivo CSV con datos de autores. Simula este escenario creando un pequeño DataFrame `df_autores` con los nombres de los autores y su año de nacimiento (inventa los datos). Luego, haz un `merge` (join) con `df_quotes` para crear un nuevo DataFrame que muestre la cita, el autor y su año de nacimiento. Pista: Usa `pd.merge()`.

In [ ]:
# Escribe aquí tu solución para el Ejercicio 4
import pandas as pd

# Datos de ejemplo
# ...

# --- INICIO DE TU CÓDIGO ---
# Crear df_autores (puedes usar df_quotes['author'].unique() para obtener los nombres únicos)
# ...
# Hacer el merge
# ...

# --- FIN DE TU CÓDIGO ---

---
## **Conclusiones y Reflexiones**

En esta sesión conceptual hemos aprendido a:

1.  **Extraer** datos no estructurados (HTML) mediante **web scraping**, transformándolos en datos tabulares.
2.  **Consumir** datos estructurados (JSON) a través de **APIs REST**, el método estándar y más limpio de intercambio de información.
3.  **Almacenar** los datos capturados en un DataFrame y persistirlos en una base de datos **SQLite**, un paso crucial para cualquier pipeline de datos.

### **Preguntas para la Reflexión:**
*   ¿En qué situaciones sería más apropiado usar web scraping en lugar de una API, y viceversa?
*   ¿Qué implicaciones éticas y legales tiene el web scraping? ¿Siempre es correcto extraer datos públicos?
*   ¿Cómo escalarían estas técnicas para capturar millones de registros por minuto para un modelo de IA? (Pista: piensa en herramientas como Apache Kafka o Spark, que veremos en sesiones futuras).

En el próximo notebook (NB2), aplicaremos estos conceptos con herramientas online y datasets reales para construir un pequeño pipeline de captura de datos.